# Part 3 — Constrained Decoding

Constrained (Hebrew-only) decoding for `Qwen/Qwen2.5-7B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3`.

**Run first (from terminal):**
```
python code/part3/run_part3.py
```
That script produces (in the workspace root):
- `hebrew_allowed_tokens_qwen.json`
- `hebrew_allowed_tokens_mistral.json`
- `decoding_outputs.jsonl`

This notebook loads those files and displays the results.

## 3.1 — Hebrew Token Identification

In [ ]:
import json
import pandas as pd

TOKEN_FILES = {
    "Qwen/Qwen2.5-7B-Instruct":           "../../hebrew_allowed_tokens_qwen.json",
    "mistralai/Mistral-7B-Instruct-v0.3": "../../hebrew_allowed_tokens_mistral.json",
}

token_data = {}
for model_id, path in TOKEN_FILES.items():
    with open(path, encoding="utf-8") as f:
        token_data[model_id] = json.load(f)

rows = []
for model_id, d in token_data.items():
    rows.append({
        "model_id":      model_id,
        "allowed_count": len(d["allowed_token_ids"]),
    })

df_tokens = pd.DataFrame(rows)
df_tokens

## 3.2 — Inference Results

In [ ]:
records = []
with open("../../decoding_outputs.jsonl", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Loaded {len(records)} records  ({df['model'].nunique()} models × {df['prompt'].nunique()} queries)")
df[["model", "prompt"]]

In [ ]:
pd.set_option("display.max_colwidth", 120)

for i, query in enumerate(df["prompt"].unique(), 1):
    print(f"\n{'='*70}")
    print(f"Query {i}: {query}")
    print('='*70)
    for _, row in df[df["prompt"] == query].iterrows():
        short = row["model"].split("/")[-1]
        print(f"\n  [{short}] Unconstrained:")
        print(f"    {row['unconstrained_output'][:400]}")
        print(f"\n  [{short}] Constrained (Hebrew-only):")
        print(f"    {row['constrained_output'][:400]}")